In [1]:
from pathlib import Path
import zipfile
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

In [2]:
#setting up path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
EX36_DIR = PROJECT_ROOT / "data" / "exercise_3_6"

TEST_ZIP = RAW_DIR / "test.zip"
TEST_DIR = EX36_DIR / "test"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(exist_ok=True)

print("Test zip exists:", TEST_ZIP.exists())
print("Models folder exists:", MODELS_DIR.exists())

print("\nModel files:")
for file in MODELS_DIR.glob("*.pt"):
    print(file.name)

Test zip exists: True
Models folder exists: True

Model files:
has_vehicle_resnet18.pt
has_pedestrian_resnet18.pt
has_traffic_light_resnet18.pt


In [3]:
#extract test data
def unzip_once(zip_path, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)

    if any(output_dir.iterdir()):
        print(f"Already extracted, skipping: {output_dir}")
        return

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)

    print(f"Extracted {zip_path.name} to {output_dir}")

unzip_once(TEST_ZIP, TEST_DIR)

#load test labels
test_csv = sorted(TEST_DIR.rglob("*.csv"))[0]
test_df = pd.read_csv(test_csv)

print("Test shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

display(test_df.head())

Already extracted, skipping: /Users/niveditakowlagi/Documents/DKE/SoSe 2026/Intro to ML/ML_Safety_Exercise3/data/exercise_3_6/test
Test shape: (3600, 7)
Columns: ['frame', 'has_traffic_light', 'has_pedestrian', 'has_vehicle', 'px_traffic_light', 'px_pedestrian', 'px_vehicle']


,frame,has_traffic_light,has_pedestrian,has_vehicle,px_traffic_light,px_pedestrian,px_vehicle
0,0,False,False,False,15,0,35
1,10,True,False,True,299,0,116
2,20,True,False,True,298,0,307
3,30,True,False,True,297,0,258
4,40,True,False,True,297,0,249


In [4]:
#image mapping prep
IMAGE_EXTENSIONS = ["*.png", "*.jpg", "*.jpeg"]

def find_images(folder):
    image_files = []
    for ext in IMAGE_EXTENSIONS:
        image_files.extend(folder.rglob(ext))
    return image_files

def build_frame_to_image_map(folder):
    image_files = find_images(folder)
    frame_to_image = {}

    for image_path in image_files:
        number = "".join([c for c in image_path.stem if c.isdigit()])
        if number != "":
            frame_to_image[int(number)] = image_path

    return frame_to_image

test_frame_to_image = build_frame_to_image_map(TEST_DIR)

print("Mapped test images:", len(test_frame_to_image))
print("First 5 frames:", list(test_frame_to_image.keys())[:5])

Mapped test images: 3600
First 5 frames: [31600, 35300, 27710, 23210, 26340]


In [5]:
#define dataset clas
class CarlaBinaryDataset(Dataset):
    def __init__(self, dataframe, frame_to_image, label_column, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.frame_to_image = frame_to_image
        self.label_column = label_column
        self.transform = transform

        self.df = self.df[
            self.df["frame"].astype(int).isin(frame_to_image.keys())
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frame = int(row["frame"])

        image_path = self.frame_to_image[frame]
        image = Image.open(image_path).convert("RGB")

        label = torch.tensor(float(row[self.label_column]), dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

#image transformation as training
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

#recreate the model architechture (weights=None, because we are loading saved trained weights)
def create_model():
    model = models.resnet18(weights=None)

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 1)

    return model


In [6]:
#evaluation function
def evaluate_model(label_column, model_path, batch_size=32):
    print(f"\nEvaluating: {label_column}")

    dataset = CarlaBinaryDataset(
        test_df,
        test_frame_to_image,
        label_column,
        transform=image_transform
    )

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print("Using device:", device)

    model = create_model().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    all_true = []
    all_pred = []

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(device)

            logits = model(images)
            probabilities = torch.sigmoid(logits).squeeze(1)

            predictions = (probabilities >= 0.5).int().cpu().numpy()
            labels = labels.int().cpu().numpy()

            all_pred.extend(predictions)
            all_true.extend(labels)

    accuracy = accuracy_score(all_true, all_pred)
    precision = precision_score(all_true, all_pred, zero_division=0)
    recall = recall_score(all_true, all_pred, zero_division=0)
    f1 = f1_score(all_true, all_pred, zero_division=0)

    return {
        "label": label_column,
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4)
    }

In [7]:
#evaluate all 3 models
# 3.6.1. Report accuracy, precision, recall, and F1-score for each model.
MODEL_PATHS = {
    "has_pedestrian": MODELS_DIR / "has_pedestrian_resnet18.pt",
    "has_traffic_light": MODELS_DIR / "has_traffic_light_resnet18.pt",
    "has_vehicle": MODELS_DIR / "has_vehicle_resnet18.pt"
}

results = []

for label_column, model_path in MODEL_PATHS.items():
    result = evaluate_model(label_column, model_path)
    results.append(result)

results_df = pd.DataFrame(results)

display(results_df)

#save results
results_path = RESULTS_DIR / "exercise_3_6_1_evaluation_results.csv"
results_df.to_csv(results_path, index=False)

print("Saved results to:", results_path)


Evaluating: has_pedestrian
Using device: mps


100%|██████████| 113/113 [00:17<00:00,  6.58it/s]



Evaluating: has_traffic_light
Using device: mps


100%|██████████| 113/113 [00:17<00:00,  6.29it/s]



Evaluating: has_vehicle
Using device: mps


100%|██████████| 113/113 [00:19<00:00,  5.72it/s]


,label,accuracy,precision,recall,f1_score
0,has_pedestrian,0.8206,0.6389,0.1955,0.2993
1,has_traffic_light,0.9422,0.9473,0.9737,0.9603
2,has_vehicle,0.8756,0.9615,0.8689,0.9128


Saved results to: /Users/niveditakowlagi/Documents/DKE/SoSe 2026/Intro to ML/ML_Safety_Exercise3/results/exercise_3_6_1_evaluation_results.csv


In [12]:
#checking the worst model
worst_model = results_df.sort_values("f1_score").iloc[0]

print("Worst model based on F1-score:")
display(worst_model)

Worst model based on F1-score:


label        has_pedestrian
accuracy             0.8206
precision            0.6389
recall               0.1955
f1_score             0.2993
Name: 0, dtype: object

2. Which model performs worst? Hypothesize why
The pedestrian detection algorithm has performed poorly, because it has F1 score of just 0.2993 and a Recall (sensitivity) of only 0.1955. This indicates that many pedestrians were missed during detection. One possible explanation for this may be due to their low abundance within the dataset, making them a smaller target and more difficult to identify from an image.

3. From a safety perspective, which metric matters most for each model– precision or recall? Explain.
The most important aspect of safety is recall, because the consequences of missing a true object are more serious than those resulting from a false detection. This is relevant to pedestrians where the recall Failure would result in the failure to determine the presence of a pedestrian. The failure to recall objects would generally lead to the failure to identify objects with which to collide for vehicles, and for traffic signals, it could lead to the failure to identify critical traffic signals. Precision is also important, but recall is the priority in this safety setting.

